# 01. TransformerLens Lecture Lab

## 강의 목표

이 노트북을 끝내면 다음을 할 수 있어야 한다.

- prompt가 token/logits로 바뀌는 과정을 설명한다.
- `cache` 안의 activation 이름과 shape를 읽는다.
- attention pattern을 시각화하되, 그것이 곧 explanation은 아님을 안다.
- zero ablation으로 특정 activation의 causal effect를 측정한다.
- clean/corrupt prompt로 activation patching의 기본 구조를 이해한다.

> 이 노트북은 “정답 코드”가 아니라 강의자료 + 실습지다. TODO를 직접 채우는 게 핵심이다.

## 0. Setup

기존 노트북에서 이미 `TransformerBridge`를 쓰고 있으므로 같은 스타일로 시작한다.

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt

try:
    from transformer_lens.model_bridge import TransformerBridge
    print("TransformerBridge import OK")
except Exception as e:
    print("Import failed:", repr(e))
    print("현재 venv/kernel이 기존 transformerlens.ipynb와 같은지 확인할 것")

## 1. Load Model

처음에는 GPT-2 small로 충분하다. CPU에서도 느리지만 기본 실험은 가능하다.

모델을 “문장 생성기”가 아니라 **activation producing machine**으로 본다.

In [ ]:
bridge = TransformerBridge.boot_transformers("gpt2", device="cpu")
print("model loaded")

## 2. Prompt → Tokens → Logits

- `tokens`: integer ids, shape는 보통 `[batch, position]`
- `str_tokens`: 사람이 읽기 쉬운 token string
- `logits`: 각 position에서 다음 token vocabulary score, shape는 `[batch, position, vocab]`

마지막 position의 logits를 보면 “다음 token으로 무엇을 예측하는지”를 알 수 있다.

In [ ]:
prompt = "The capital of France is"

tokens = bridge.to_tokens(prompt)
str_tokens = bridge.to_str_tokens(prompt)
logits, cache = bridge.run_with_cache(tokens)

print("prompt:", prompt)
print("tokens shape:", tokens.shape)
print("str_tokens:", str_tokens)
print("logits shape:", logits.shape)

### Demo: Top-k next token

아래 결과를 보고 모델이 다음 token으로 무엇을 예상하는지 확인한다.

In [ ]:
next_logits = logits[0, -1]
topk = torch.topk(next_logits, k=10)

rows = []
for rank, (token_id, logit) in enumerate(zip(topk.indices, topk.values), start=1):
    rows.append({
        "rank": rank,
        "token_id": int(token_id),
        "token": repr(bridge.to_string(int(token_id))),
        "logit": float(logit),
    })

pd.DataFrame(rows)

### Exercise 1

아래 prompt를 바꿔가며 top-k가 어떻게 바뀌는지 본다.

추천 prompt:

- `The capital of Korea is`
- `The opposite of hot is`
- `John and Mary went to the store. John gave a book to`

체크포인트:

- [ ] token 앞 공백이 prediction에 어떤 영향을 주는지 확인했다.
- [ ] final position logits만 보는 이유를 설명할 수 있다.

In [ ]:
def show_topk(prompt, k=10):
    tokens = bridge.to_tokens(prompt)
    logits, cache = bridge.run_with_cache(tokens)
    next_logits = logits[0, -1]
    topk = torch.topk(next_logits, k=k)
    rows = []
    for rank, (token_id, logit) in enumerate(zip(topk.indices, topk.values), start=1):
        rows.append({"rank": rank, "token": repr(bridge.to_string(int(token_id))), "logit": float(logit)})
    return pd.DataFrame(rows)

show_topk("The opposite of hot is")

## 3. Cache: 내부 activation 지도 보기

`run_with_cache`는 forward pass 중간 activation을 저장한다. Mech interp의 시작은 cache 이름과 shape를 읽는 것이다.

자주 보는 축: batch / position-token / head / d_model / d_head / d_mlp

In [ ]:
for name, act in list(cache.items())[:30]:
    print(name, tuple(act.shape))

### Exercise 2: 주요 activation 표 만들기

각 row 옆에 “이 activation이 무엇을 의미하는지”를 markdown으로 직접 적어라.

In [ ]:
interesting_prefixes = ["embed", "pos_embed", "blocks.0", "blocks.1", "ln_final"]
rows = []
for name, act in cache.items():
    if name.startswith(tuple(interesting_prefixes)):
        rows.append({"name": name, "shape": tuple(act.shape), "ndim": act.ndim})

pd.DataFrame(rows).head(50)

## 4. Attention Pattern 시각화

Attention pattern은 “어느 token을 봤는지”를 보여준다. 하지만 이것만으로 “왜 답이 나왔는지”를 증명하지는 못한다.

> Static clue이지 causal evidence가 아니다.

In [ ]:
layer = 0
head = 0
pattern = cache[f"blocks.{layer}.attn.hook_pattern"]
if pattern.ndim == 4:
    pattern = pattern[0]

plt.figure(figsize=(7, 5))
plt.imshow(pattern[head].detach().cpu(), cmap="Blues")
plt.xticks(range(len(str_tokens)), str_tokens, rotation=45, ha="right")
plt.yticks(range(len(str_tokens)), str_tokens)
plt.title(f"Layer {layer} Head {head} Attention Pattern")
plt.colorbar()
plt.tight_layout()
plt.show()

### Exercise 3

다른 layer/head를 바꿔보면서 흥미로운 pattern 3개를 찾는다.

기록할 것:

- layer/head
- 어떤 token을 주로 보는가?
- 이게 출력에 중요하다고 말하려면 어떤 intervention이 필요한가?

In [ ]:
layer = 5
head = 3
pattern = cache[f"blocks.{layer}.attn.hook_pattern"]
if pattern.ndim == 4:
    pattern = pattern[0]

plt.figure(figsize=(7, 5))
plt.imshow(pattern[head].detach().cpu(), cmap="Blues")
plt.xticks(range(len(str_tokens)), str_tokens, rotation=45, ha="right")
plt.yticks(range(len(str_tokens)), str_tokens)
plt.title(f"Layer {layer} Head {head} Attention Pattern")
plt.colorbar()
plt.tight_layout()
plt.show()

## 5. Metric: Logit Difference

Intervention을 하려면 metric이 필요하다.

예: `The capital of France is`에서 ` Paris`가 ` London`보다 얼마나 높은지 본다.

`logit_diff = logit(correct) - logit(wrong)`

In [ ]:
def single_token_id(s):
    return bridge.to_tokens(s)[0, -1].item()

correct_id = single_token_id(" Paris")
wrong_id = single_token_id(" London")

base_logits = bridge(prompt)
base_diff = base_logits[0, -1, correct_id] - base_logits[0, -1, wrong_id]
print("logit diff Paris-London:", float(base_diff))

## 6. Zero Ablation

특정 activation을 0으로 만들고 metric이 얼마나 변하는지 본다. 이게 첫 causal test다.

In [ ]:
def zero_ablate_hook(act, hook):
    return torch.zeros_like(act)

clean_logits = bridge(prompt)
ablated_logits = bridge.run_with_hooks(
    prompt,
    fwd_hooks=[("blocks.0.attn.hook_out", zero_ablate_hook)]
)

clean_diff = clean_logits[0, -1, correct_id] - clean_logits[0, -1, wrong_id]
ablated_diff = ablated_logits[0, -1, correct_id] - ablated_logits[0, -1, wrong_id]

pd.DataFrame([
    {"condition": "clean", "logit_diff": float(clean_diff)},
    {"condition": "ablate blocks.0.attn.hook_out", "logit_diff": float(ablated_diff)},
    {"condition": "delta", "logit_diff": float(ablated_diff - clean_diff)},
])

### Exercise 4: layer별 ablation scan

아래 코드는 layer별 `attn.hook_out` ablation 결과를 표/그래프로 만든다.

체크포인트:

- [ ] 어떤 layer를 지웠을 때 metric이 가장 크게 변하는가?
- [ ] 그 결과가 attention pattern 관찰과 일치하는가?

In [ ]:
results = []
for layer in range(12):
    hook_name = f"blocks.{layer}.attn.hook_out"
    ablated_logits = bridge.run_with_hooks(prompt, fwd_hooks=[(hook_name, zero_ablate_hook)])
    diff = ablated_logits[0, -1, correct_id] - ablated_logits[0, -1, wrong_id]
    results.append({"layer": layer, "hook": hook_name, "logit_diff": float(diff), "delta": float(diff - clean_diff)})

ablation_df = pd.DataFrame(results)
ablation_df

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(ablation_df["layer"], ablation_df["delta"])
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("Layer")
plt.ylabel("Delta logit diff after ablation")
plt.title("Layer-wise attention output ablation")
plt.show()

## 7. Activation Patching 개념

Activation patching은 clean run의 activation을 corrupt run에 이식해서 metric이 복구되는지 보는 방법이다.

절차:

1. clean prompt: 모델이 정답을 맞추는 prompt
2. corrupt prompt: 중요한 정보가 바뀐 prompt
3. clean cache 저장
4. corrupt forward 중 특정 activation만 clean activation으로 교체
5. answer logit diff가 얼마나 복구되는지 측정

In [ ]:
clean_prompt = "After John and Mary went to the store, Mary gave a bottle of milk to"
corrupt_prompt = "After John and Mary went to the store, John gave a bottle of milk to"

clean_tokens = bridge.to_tokens(clean_prompt)
corrupt_tokens = bridge.to_tokens(corrupt_prompt)

print("clean tokens:", bridge.to_str_tokens(clean_prompt))
print("corrupt tokens:", bridge.to_str_tokens(corrupt_prompt))
print("same shape?", clean_tokens.shape == corrupt_tokens.shape)

In [ ]:
john_id = single_token_id(" John")
mary_id = single_token_id(" Mary")

clean_logits, clean_cache = bridge.run_with_cache(clean_prompt)
corrupt_logits, corrupt_cache = bridge.run_with_cache(corrupt_prompt)

def ioi_metric(logits):
    return logits[0, -1, john_id] - logits[0, -1, mary_id]

print("clean metric:", float(ioi_metric(clean_logits)))
print("corrupt metric:", float(ioi_metric(corrupt_logits)))

In [ ]:
from functools import partial

def patch_resid_hook(resid, hook, position):
    clean_resid = clean_cache[hook.name]
    resid[:, position, :] = clean_resid[:, position, :]
    return resid

layer = 7
position = -1
hook_name = f"blocks.{layer}.hook_resid_post"

patched_logits = bridge.run_with_hooks(
    corrupt_prompt,
    fwd_hooks=[(hook_name, partial(patch_resid_hook, position=position))]
)

print("patched metric:", float(ioi_metric(patched_logits)))

### Exercise 5: layer x position patching heatmap

아래를 직접 구현해라.

- rows: layer 0~11
- cols: position 0~sequence length-1
- value: patched metric 또는 normalized recovery

완료 기준:

- [ ] heatmap 생성
- [ ] 가장 높은 위치 3개 기록
- [ ] 그 위치를 다시 ablation해서 교차 검증

In [ ]:
# TODO: layer x position patching scan 구현
# seq_len = clean_tokens.shape[1]
# scores = torch.zeros((12, seq_len))
# for layer in range(12):
#   for position in range(seq_len):
#       hook_name = f"blocks.{layer}.hook_resid_post"
#       patched_logits = bridge.run_with_hooks(...)
#       scores[layer, position] = float(ioi_metric(patched_logits))

print("TODO: implement patching heatmap")

## 8. Writeup Prompt

`notes/week-01.md` 또는 별도 README에 아래 질문에 답해라.

1. 내가 정의한 metric은 무엇인가?
2. attention pattern에서 흥미로웠던 head는 무엇인가?
3. ablation 결과와 attention 관찰은 일치했나?
4. activation patching에서 가장 중요해 보인 layer/position은 어디인가?
5. 이 결과의 한계는 무엇인가?